In [2]:
# For HF batch embedding in this notebook, the server API is used.
# Ensure HF_API_TOKEN is set in environment:
#   setx HF_API_TOKEN "your_token"

c:\Users\Loq\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
c:\Users\Loq\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
c:\Users\Loq\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/res

In [5]:
import json
import os
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

HF_EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
HF_API_TOKEN = os.environ.get('HF_API_TOKEN')  # set your Hugging Face token, e.g., os.environ.get('HF_API_TOKEN')


def _mean_pool_embedding(raw_embedding):
    if not raw_embedding:
        raise ValueError("Hugging Face API returned an empty embedding.")

    if isinstance(raw_embedding[0], (int, float)):
        return [float(value) for value in raw_embedding]

    if isinstance(raw_embedding[0], list):
        token_count = len(raw_embedding)
        vector_size = len(raw_embedding[0])
        pooled = [0.0] * vector_size

        for token_vector in raw_embedding:
            if len(token_vector) != vector_size:
                raise ValueError("Inconsistent token vector dimensions in HF embedding response.")
            for idx, value in enumerate(token_vector):
                pooled[idx] += float(value)

        return [value / token_count for value in pooled]

    raise ValueError("Unexpected Hugging Face embedding response format.")


def get_embedding(text, model_name: str | None = None):
    selected_model = model_name or HF_EMBEDDING_MODEL
    endpoint = (
        "https://router.huggingface.co/hf-inference/models/"
        f"{selected_model}/pipeline/feature-extraction"
    )
    payload = json.dumps({"inputs": text, "normalize": True}).encode("utf-8")

    headers = {"Content-Type": "application/json"}
    if HF_API_TOKEN:
        headers["Authorization"] = f"Bearer {HF_API_TOKEN}"

    request = Request(endpoint, data=payload, headers=headers, method="POST")

    try:
        with urlopen(request, timeout=60) as response:
            response_data = json.loads(response.read().decode("utf-8"))
    except HTTPError as exc:
        message = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(
            f"Hugging Face embedding API request failed ({exc.code}): {message}"
        ) from exc
    except URLError as exc:
        raise RuntimeError(f"Could not reach Hugging Face embedding API: {exc}") from exc

    if isinstance(response_data, dict) and response_data.get("error"):
        raise RuntimeError(f"Hugging Face embedding API error: {response_data['error']}")

    if isinstance(response_data, list) and len(response_data) == 1:
        return _mean_pool_embedding(response_data[0])

    return _mean_pool_embedding(response_data)


def get_embedding_batch(text_list, batch_size=100, model_name: str | None = None):
    if not isinstance(text_list, list):
        raise ValueError("text_list must be a list of strings")
    if len(text_list) == 0:
        return []

    embeddings = []
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i : i + batch_size]
        endpoint = (
            "https://router.huggingface.co/hf-inference/models/"
            f"{(model_name or HF_EMBEDDING_MODEL)}/pipeline/feature-extraction"
        )
        payload = json.dumps({"inputs": batch, "normalize": True}).encode("utf-8")

        headers = {"Content-Type": "application/json"}
        if HF_API_TOKEN:
            headers["Authorization"] = f"Bearer {HF_API_TOKEN}"

        request = Request(endpoint, data=payload, headers=headers, method="POST")

        try:
            with urlopen(request, timeout=120) as response:
                response_data = json.loads(response.read().decode("utf-8"))
        except HTTPError as exc:
            message = exc.read().decode("utf-8", errors="replace")
            raise RuntimeError(
                f"Hugging Face embedding API request failed ({exc.code}): {message}"
            ) from exc
        except URLError as exc:
            raise RuntimeError(f"Could not reach Hugging Face embedding API: {exc}") from exc

        if isinstance(response_data, dict) and response_data.get("error"):
            raise RuntimeError(f"Hugging Face embedding API error: {response_data['error']}")

        if not isinstance(response_data, list):
            raise ValueError("Unexpected response format from HF API for batch embedding")

        for raw in response_data:
            embeddings.append(_mean_pool_embedding(raw))

    return embeddings


# Test for 1,000 entries
if __name__ == "__main__":
    sample_texts = [f"Test sentence {i}" for i in range(1000)]
    print("Sending batch embedding request for 1000 sentences...")
    result = get_embedding_batch(sample_texts, batch_size=100)
    assert len(result) == 1000, f"expected 1000 embeddings, got {len(result)}"
    assert all(len(v) > 0 for v in result), "some embeddings were empty"
    print("Finished test: 1000 embeddings returned successfully")

Sending batch embedding request for 1000 sentences...
Finished test: 1000 embeddings returned successfully
